**<h1 style="text-align: center;font-size: 3rem">Onnx Loading</h1><p style="text-align: center;font-size: 1.3rem">(Notebook IV)</p>**


## Imports

Getting Microsoft's packages for data analysis and ML with Onnx runtime via NuGet to emulate predictions using the exported Onnx model file. Then they are defined.


In [ ]:
#r "nuget: Microsoft.Data.Analysis"
#r "nuget: Microsoft.ML"
#r "nuget: Microsoft.ML.OnnxRuntime"

using Microsoft.ML.OnnxRuntime;
using Microsoft.ML.OnnxRuntime.Tensors;
using Microsoft.ML.Data;
using Microsoft.Data.Analysis;
using System.IO;
using System.Linq;
using System.Globalization;
using System.Text.Json;
using System.Text.Json.Serialization;

## Variables

Paths to the model, dataset, and column schema.


In [ ]:
var modelPath = "../models/fraud_detection_xgbclassifier.onnx";
var dataPath = "../data/processed/testdata.csv";
var schemaPath = "../models/columns_schema.json";

## Utility Setup

### ColumnSchema

ColumnSchema will be a class to deserialize the JSON schema file so that the dataframe in C# stores the data in the same positions that python does.


In [ ]:
public class ColumnSchema
{
    [JsonPropertyName("feature_names")]
    public string[] FeatureNames { get; set; }

    [JsonPropertyName("target_name")]
    public string TargetName { get; set; }

    public string[] Columns => FeatureNames.Append(TargetName).ToArray();
}

### Metrics

Metrics is a static class that contains methods for calculating common metrics to check comparable results to the model running in python.


In [ ]:
public static class Metrics
{
    private static int ZipCompareLabels(IEnumerable<int> actualLabel, IEnumerable<int> predictedLabel, Func<int, int, int> comparer)
    {
        return actualLabel.Zip(predictedLabel, comparer).Sum();
    }

    private static int GetTruePositives(IEnumerable<int> actualLabel, IEnumerable<int> predictedLabel)
    {
        return ZipCompareLabels(actualLabel, predictedLabel, (t, p) => t == 1 && p == 1 ? 1 : 0);
    }

    private static int GetTrueNegatives(IEnumerable<int> actualLabel, IEnumerable<int> predictedLabel)
    {
        return ZipCompareLabels(actualLabel, predictedLabel, (t, p) => t == 0 && p == 0 ? 1 : 0);
    }

    private static int GetFalsePositives(IEnumerable<int> actualLabel, IEnumerable<int> predictedLabel)
    {
        return ZipCompareLabels(actualLabel, predictedLabel, (t, p) => t == 0 && p == 1 ? 1 : 0);
    }

    private static int GetFalseNegatives(IEnumerable<int> actualLabel, IEnumerable<int> predictedLabel)
    {
        return ZipCompareLabels(actualLabel, predictedLabel, (t, p) => t == 1 && p == 0 ? 1 : 0);
    }

    public static double Accuracy(IEnumerable<int> actualLabel, IEnumerable<int> predictedLabel)
    {
        return (GetTruePositives(actualLabel, predictedLabel) + GetTrueNegatives(actualLabel, predictedLabel))
            / (double)(actualLabel.Count());
    }

    public static double Precision(IEnumerable<int> actualLabel, IEnumerable<int> predictedLabel)
    {
        var tp = GetTruePositives(actualLabel, predictedLabel);
        var fp = GetFalsePositives(actualLabel, predictedLabel);
        return tp / (double)(tp + fp);
    }

    public static double Recall(IEnumerable<int> actualLabel, IEnumerable<int> predictedLabel)
    {
        var tp = GetTruePositives(actualLabel, predictedLabel);
        var fn = GetFalseNegatives(actualLabel, predictedLabel);
        return tp / (double)(tp + fn);
    }

    public static double F1Score(IEnumerable<int> actualLabel, IEnumerable<int> predictedLabel)
    {
        var precision = Precision(actualLabel, predictedLabel);
        var recall = Recall(actualLabel, predictedLabel);
        return 2 * (precision * recall) / (precision + recall);
    }
}

### Other Methods

The below methods are add for transforming the data and converting the dataframe rows into batched tensors.


In [ ]:
public static (double, double) ToCyclicFeature(int elapsedTime)
{
    double period = 24 * 60 * 60; // seconds in a day
    double angle = 2 * Math.PI * elapsedTime / period;
    return (Math.Sin(angle), Math.Cos(angle));
}

public static IEnumerable<DenseTensor<float>> TensorBatchesFromDataFrame(DataFrame df, int batchSize)
{
    var size = Math.Max(1, batchSize);
    var batch = new List<float[]>(size);
    
    foreach (var row in df.Rows)
    {
        var floats = new float[df.Columns.Count];
        for (int i = 0; i < df.Columns.Count; i++)
        {
            floats[i] = Convert.ToSingle(row[i]);
        }
        batch.Add(floats);

        if (batch.Count == size)
        {
            yield return ToTensor(batch);
            batch.Clear();
        }
    }
}

public static DenseTensor<float> ToTensor(IList<float[]> batch)
{
    if (batch is null || batch.Count == 0) throw new ArgumentException("Batch is empty");

    int batchSize = batch.Count;
    int featureCount = batch[0].Length;
    var tensor = new DenseTensor<float>(new[] { batchSize, featureCount });

    for (int i = 0; i < batchSize; i++)
    {
        for (int j = 0; j < featureCount; j++)
        {
            tensor[i, j] = batch[i][j];
        }
    }

    return tensor;
}

In [ ]:
string jsonString = File.ReadAllText(schemaPath);

ColumnSchema schema = JsonSerializer.Deserialize<ColumnSchema>(jsonString);

Console.WriteLine("Loaded schema with features: " + string.Join(", ", schema.Columns));

In [ ]:
DataFrame df = DataFrame.LoadCsv(dataPath); 
df = new DataFrame(schema.Columns.Select(name => df.Columns[name]).ToArray());

In [ ]:
DataFrame featureMatrix = new DataFrame(schema.FeatureNames.Select(name => df[name]).ToArray());

var fraudColumn = ((SingleDataFrameColumn) df["is_fraud"]).Select(v => v.HasValue && v.Value != 0f ? 1 : 0);

In [ ]:
var stream = TensorBatchesFromDataFrame(featureMatrix, 512);

In [ ]:
var session = new InferenceSession(modelPath);

In [ ]:
var inputs = session.InputMetadata.Keys.ToList();
Console.WriteLine($"Model input inputs: {string.Join(", ", inputs)}");

var outputs = session.OutputMetadata.Keys.ToList();
Console.WriteLine($"Model output outputs: {string.Join(", ", outputs)}");

In [ ]:
List<int> preds = [];

foreach (var inputBatch in TensorBatchesFromDataFrame(featureMatrix, 512))
{
    var results = session.Run(new[] { NamedOnnxValue.CreateFromTensor("input", inputBatch) });
    var t = results[0].AsTensor<Int64>();
    preds.AddRange(t.Select(v => (int) v).ToArray());
}

In [ ]:
Console.WriteLine($"Total predictions: {preds.Count}");
Console.WriteLine($"Accuracy: {Metrics.Accuracy(fraudColumn, preds)}");
Console.WriteLine($"Precision: {Metrics.Precision(fraudColumn, preds)}");
Console.WriteLine($"Recall: {Metrics.Recall(fraudColumn, preds)}");
Console.WriteLine($"F1 Score: {Metrics.F1Score(fraudColumn, preds)}");